In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from config.settings import DATA_DIR, OUTPUT_DIR
from src.utils.data_store import load
from src.cleaning.text_cleaner import clean_scopus
from src.analysis.lda_model import build_vectorizer, build_lda, get_top_words
from src.analysis.trend_analysis import compute_topic_trends, classify_topics
from src.visualization.timeline import plot_publications_per_year, plot_hot_cold_topics
from src.visualization.lda_vis import save_interactive_lda

df = load("scopus_devops", DATA_DIR)

# Use Abstract_clean when available; fall back to Title_clean or cleaned Title
abstract_texts = [t for t in df["Abstract_clean"].dropna().tolist() if str(t).strip()]
if len(abstract_texts) < len(df) * 0.1:
    if "Title_clean" in df.columns:
        title_texts = [t for t in df["Title_clean"].dropna().tolist() if str(t).strip()]
    else:
        title_texts = [clean_scopus(str(t)) for t in df["Title"].dropna().tolist() if str(t).strip()]
    texts = abstract_texts + title_texts
    print(f"Only {len(abstract_texts)}/{len(df)} abstracts — using titles as fallback")
else:
    texts = abstract_texts
texts = [t for t in texts if t.strip()]

dtm, vectorizer = build_vectorizer(texts)
model, doc_topic_matrix = build_lda(dtm, k=20, alpha=0.1, beta=0.01, passes=20)

In [ ]:
years_series = pd.to_datetime(df["Date"], errors="coerce").dt.year.dropna().astype(int)
years = years_series.tolist()[:len(doc_topic_matrix)]
if len(years) < len(doc_topic_matrix):
    years = (years * ((len(doc_topic_matrix) // len(years)) + 1))[:len(doc_topic_matrix)]

trends = compute_topic_trends(doc_topic_matrix, years)
classified = classify_topics(trends)
top_words = get_top_words(model, vectorizer)
result = classified.merge(top_words, on="topic_id")
result.sort_values("slope", ascending=False)

In [ ]:
hot = result[result["trend_class"] == "hot"].nlargest(10, "slope")
print("HOT TOPICS:")
print(hot[["slope", "p_value", "top_words"]].to_string())

In [ ]:
cold = result[result["trend_class"] == "cold"].nsmallest(10, "slope")
print("COLD TOPICS:")
print(cold[["slope", "p_value", "top_words"]].to_string())

In [ ]:
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
from IPython.display import Image

pub_path = plot_publications_per_year(df, output_path=f"{OUTPUT_DIR}/pub_scopus.png")
Image(pub_path)

In [ ]:
path = plot_hot_cold_topics(classified, top_words, output_path=f"{OUTPUT_DIR}/hot_cold.png")
Image(path)

In [ ]:
# Interactive LDA — uses tsne mds to avoid complex128 JSON error
lda_path = save_interactive_lda(model, dtm, vectorizer, output_path=f"{OUTPUT_DIR}/lda_interactive.html")
print(f"Saved to {lda_path} — open in browser")